# ✦ LILY WAN 2.2 — Adaptive Kaggle Studio — v7

v7 keeps the adaptive T4 x2 / P100 behavior, fixes the generated-code syntax bug, and prevents pip from recursively downloading a second Torch/CUDA stack.


In [ ]:
import json, urllib.request

print('✦ Lily Wan 2.2 Studio — startup v7')
print('✓ Loading adaptive build and applying safe installer patches...')

V5_URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/7268f44cfd9f610b156a08cc3d064b3427b60c63/LILY_WAN22_DUAL_T4_STUDIO.ipynb'
with urllib.request.urlopen(V5_URL, timeout=60) as r:
    nb = json.loads(r.read().decode('utf-8'))

cells = [c for c in nb['cells'] if c.get('cell_type') == 'code']
if not cells:
    raise RuntimeError('Could not load the adaptive studio code cell.')
wrapper = ''.join(cells[0]['source'])

# Patch 1: repair v5's generated newline source safely.
fixed_lines = []
newline_repaired = False
nodeps_patched = 0
for line in wrapper.splitlines():
    if '_req_safe.write_text(' in line:
        indent = line[:len(line) - len(line.lstrip())]
        line = indent + '_req_safe.write_text(chr(10).join(_lines) + chr(10))'
        newline_repaired = True

    # Patch 2: every pip install aimed at lily_pkgs must use --no-deps.
    # This prevents ComfyUI packages from pulling another torch/CUDA/NVIDIA stack.
    if 'pip' in line and '--target' in line and 'PKG_TARGET' in line and '--no-deps' not in line:
        line = line.replace('\"--target\"', '\"--no-deps\", \"--target\"', 1)
        nodeps_patched += 1

    fixed_lines.append(line)

wrapper = chr(10).join(fixed_lines) + chr(10)

if not newline_repaired:
    raise RuntimeError('Could not locate the generated newline bug to repair.')
if nodeps_patched < 1:
    raise RuntimeError('Could not apply the no-dependency pip safety patch.')

# Compile the wrapper itself before it can clone/download anything.
compiled = compile(wrapper, 'LILY_WAN22_V7_SAFE_WRAPPER', 'exec')
print('✓ Syntax repair applied')
print(f'✓ pip recursion disabled in {nodeps_patched} sandbox installer(s)')
print('✓ Wrapper passed compile preflight')
print('✓ Reusing Kaggle Torch/CUDA instead of downloading another copy')
print('✓ Starting adaptive Wan 2.2 setup...')
exec(compiled, globals(), globals())
